# Landsat LST → 100m Grid Mapping

## 목적
- Landsat 8/9 Level-2 데이터를 이용해 여름철(6–9월) 평균 LST 산출
- 고정된 서울 100m grid(cell_id)에 LST 값을 매핑

## 입력 데이터
- GEE Asset: seoul_grid_100m (100m grid)
- Landsat 8/9 Collection 2 L2

## 출력 데이터
- cell_id, LST_C (연도별 CSV) → **Google Drive에서 다운로드 후 `data/raw/lst/`에 저장** (전처리 전 원본)
- 이상치 제거는 `05_lst_outlier_check.ipynb`에서 수행 후 `data/processed/lst/`에 저장


In [2]:
# 기본 라이브러리
import os
import ee
import geemap
from dotenv import load_dotenv

# .env 로드
load_dotenv()

# Earth Engine 인증
ee.Authenticate()

project_id = os.getenv("EARTHENGINE_PROJECT")

# Earth Engine 초기화
ee.Initialize(project=project_id)

In [4]:
# =========================
# Global Configuration
# =========================

# 분석 대상 지역
TARGET_COUNTRY = "Republic of Korea"
TARGET_CITY = "Seoul"

# 기간 설정 (여름철)
START_MMDD = "06-01"
END_MMDD = "09-01"

# Grid 설정
GRID_SIZE = 100  # meters
PROJECTION = "EPSG:3857"

# GEE Asset (100m grid)
GRID_ASSET_ID = f"projects/{project_id}/assets/seoul_grid_100m"

In [5]:
# =========================
# Load Seoul Boundary
# =========================

gaul1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

korea = gaul1.filter(
    ee.Filter.eq("ADM0_NAME", TARGET_COUNTRY)
)

seoul = korea.filter(
    ee.Filter.eq("ADM1_NAME", TARGET_CITY)
)

seoul_geometry = seoul.geometry()

print("Seoul feature count:", seoul.size().getInfo())

Seoul feature count: 1


In [7]:
# =========================
# Load 100m Grid Asset
# =========================

grid_fc = ee.FeatureCollection(GRID_ASSET_ID)

print("Grid cell count:", grid_fc.size().getInfo())

Grid cell count: 92398


In [8]:
# =========================
# LST Conversion Function
# =========================

def add_lst_celsius(img):
    """
    Landsat Collection 2 Level-2 ST_B10
    → Land Surface Temperature (Celsius)
    """
    lst_c = (
        img.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST_C")
    )
    return img.addBands(lst_c)

In [9]:
# =========================
# Cloud Mask Function
# =========================

def mask_landsat_l2(img):
    """
    QA_PIXEL 비트 정보를 이용한 구름/눈/그림자 제거
    """
    qa = img.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 1).eq(0)   # Dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # Cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # Cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # Cloud shadow
        .And(qa.bitwiseAnd(1 << 5).eq(0))  # Snow
    )

    return img.updateMask(mask)

In [10]:
# =========================
# Target Year
# =========================

YEAR = 2025

START_DATE = f"{YEAR}-{START_MMDD}"
END_DATE = f"{YEAR}-{END_MMDD}"

print("Analysis period:", START_DATE, "~", END_DATE)

Analysis period: 2025-06-01 ~ 2025-09-01


In [11]:
# =========================
# Landsat 8/9 LST Collection
# =========================

l8 = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(seoul_geometry)
    .filter(ee.Filter.lt('CLOUD_COVER', 40))
    .map(mask_landsat_l2)
    .map(add_lst_celsius)
)

l9 = (
    ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(seoul_geometry)
    .filter(ee.Filter.lt('CLOUD_COVER', 40))
    .map(mask_landsat_l2)
    .map(add_lst_celsius)
)

landsat_lst = (
    l8.merge(l9)
    .select("LST_C")
)

print("Total LST images:", landsat_lst.size().getInfo())

Total LST images: 5


In [12]:
# =========================
# Summer Mean LST Image
# =========================

lst_mean_30m = (
    landsat_lst
    .mean()
    .clip(seoul_geometry)
    .rename("LST_C")
)

print("Summer mean LST image created.")

# LST 값 범위 확인 (서울 전체)
lst_stats = lst_mean_30m.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=seoul_geometry,
    scale=30,
    maxPixels=1e9
)

print("LST min/max (°C):", lst_stats.getInfo())

Map = geemap.Map()
Map.centerObject(seoul, 11)

Map.addLayer(
    lst_mean_30m,
    {
        "min": 20,
        "max": 40,
        "palette": ["blue", "cyan", "yellow", "orange", "red"]
    },
    "Summer Mean LST (30m)"
)

Map

Summer mean LST image created.
LST min/max (°C): {'LST_C_max': 51.79257949999999, 'LST_C_min': 15.288125900000011}


Map(center=[37.53803118323625, 127.00584943229947], controls=(WidgetControl(options=['position', 'transparent_…

In [13]:
# =========================
# Map LST to 100m Grid
# =========================

grid_with_lst = lst_mean_30m.reduceRegions(
    collection=grid_fc,
    reducer=ee.Reducer.mean(),
    scale=GRID_SIZE  # 100m
)

# grid_with_lst_clean = grid_with_lst.filter(
#     ee.Filter.notNull(["mean"])
# )

print("LST mapping to grid completed.")

LST mapping to grid completed.


In [14]:
# =========================
# Clean output columns
# =========================

def rename_mean_to_lst(feat):
    return (
        feat
        .set("LST_C", feat.getNumber("mean"))
        .select(["cell_id", "LST_C"])
    )

grid_with_lst_final = grid_with_lst.map(rename_mean_to_lst)

In [15]:
# =========================
# Coverage check
# =========================

total_cells = grid_fc.size()
lst_cells = grid_with_lst_final.size()

print("Total grid cells:", total_cells.getInfo())
print("Cells with LST:", lst_cells.getInfo())
print(
    "Coverage (%):",
    (lst_cells.divide(total_cells).multiply(100)).getInfo()
)

Total grid cells: 92398
Cells with LST: 92398
Coverage (%): 100


In [17]:
# =========================
# Export to CSV (Google Drive)
# =========================

output_dir = "../data_processed/processed/lst"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(
    output_dir,
    f"lst_{YEAR}_summer_100m.csv"
)

export_lst_csv = grid_with_lst_final.select([
    "cell_id",
    "LST_C"
])

print(f"Exporting LST CSV to {csv_path}...")

geemap.ee_export_vector(
    export_lst_csv,
    filename=csv_path
)

print("LST CSV export complete.")

Exporting LST CSV to ../data_processed/processed/lst/lst_2025_summer_100m.csv...
Generating URL ...
Please wait ...
Data downloaded to /Users/roel/Project/KT_Aivle/빅프로젝트/data_AI/data_ai/data_processed/processed/lst/lst_2025_summer_100m.csv
LST CSV export complete.
